In [21]:
import pandas as pd

DATA_PATH = r"D:\Seminar_Project\dataset\News_Category_Dataset_v3.json"

df = pd.read_json(DATA_PATH, lines=True)

df.head()

,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22


## Test Classifier

In [30]:
cols_to_drop = ['link', 'date', 'link', 'authors']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
# alternatively: df = df.drop(columns=cols_to_drop, errors='ignore')
df.head()

,headline,category,short_description
0,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...
1,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...
2,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha..."
3,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to..."
4,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...


In [31]:
df.shape

(209527, 3)

In [33]:
df.category.value_counts()
print("total unique categories:", df.category.nunique())
print("categories:", df.category.unique())


total unique categories: 42
categories: ['U.S. NEWS' 'COMEDY' 'PARENTING' 'WORLD NEWS' 'CULTURE & ARTS' 'TECH'
 'SPORTS' 'ENTERTAINMENT' 'POLITICS' 'WEIRD NEWS' 'ENVIRONMENT'
 'EDUCATION' 'CRIME' 'SCIENCE' 'WELLNESS' 'BUSINESS' 'STYLE & BEAUTY'
 'FOOD & DRINK' 'MEDIA' 'QUEER VOICES' 'HOME & LIVING' 'WOMEN'
 'BLACK VOICES' 'TRAVEL' 'MONEY' 'RELIGION' 'LATINO VOICES' 'IMPACT'
 'WEDDINGS' 'COLLEGE' 'PARENTS' 'ARTS & CULTURE' 'STYLE' 'GREEN' 'TASTE'
 'HEALTHY LIVING' 'THE WORLDPOST' 'GOOD NEWS' 'WORLDPOST' 'FIFTY' 'ARTS'
 'DIVORCE']


## Try API gen.ai.kku.ac.th

In [43]:
# Please install OpenAI SDK first: 'pip3 install openai'
import os
from openai import OpenAI

client = OpenAI(api_key=os.environ.get('LLM_API'), base_url="https://gen.ai.kku.ac.th/api/v1")

response = client.chat.completions.create(
    model="gemini-2.5-flash-lite",
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": "What is the meaning of life?"},
    ],
    stream=False
)

print(response.choices[0].message.content)

Ah, the age-old question! The "meaning of life" is one of the most profound, debated, and personal inquiries humanity grapples with.

Here's the fascinating truth: **There is no single, universally agreed-upon answer.**

The meaning of life is approached and defined very differently across philosophy, religion, science, and individual experience. Here are the main perspectives on what the meaning of life might be:

---

## 1. Philosophical Perspectives

Philosophy offers a wide spectrum of answers, often revolving around happiness, virtue, or knowledge:

*   **Nihilism:** This view suggests that life is *without* objective meaning, purpose, or intrinsic value. Life simply is, and there is no inherent reason for it.
*   **Existentialism (Sartre, de Beauvoir):** Existentialists argue that "existence precedes essence." We are born without a pre-defined purpose, and it is our fundamental freedom and responsibility to **create our own meaning** through our choices, actions, and commitments.

In [41]:
from dotenv import load_dotenv
load_dotenv()

True

In [69]:
client = OpenAI(
    api_key=os.environ.get('LLM_API'),
    base_url="https://gen.ai.kku.ac.th/api/v1"
)
MODEL = "deepseek-v3.2"

In [70]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

import numpy as np
import time

In [71]:
N_SAMPLES_PER_CLASS = 7
RANDOM_SEED = 42

In [72]:
def load_data(path, n_per_class=N_SAMPLES_PER_CLASS, seed=RANDOM_SEED):
    print(f"Loading dataset from:\n  {path}")
    df = pd.read_json(path, lines=True)
    print(f"Total records : {len(df):,}")
    print(f"Columns       : {df.columns.tolist()}")
    print(f"Categories    : {df['category'].nunique()} classes")
    print(f"  {sorted(df['category'].unique())}\n")

    # ลบ rows ที่ข้อมูลสำคัญว่าง
    df = df.dropna(subset=['headline', 'short_description', 'category'])
    df = df[df['short_description'].str.strip() != '']
    df = df[df['headline'].str.strip() != '']

    # Stratified sample: เอา n_per_class*3 ต่อ class
    # (train จะได้ ~70%, test ~30%)
    df_sampled = (
        df.groupby('category', group_keys=False)
        .apply(lambda x: x.sample(min(len(x), n_per_class * 3), random_state=seed))
        .reset_index(drop=True)
    )

    print(f"Sampled records: {len(df_sampled):,} ({n_per_class*3} per class max)\n")
    return df_sampled

In [73]:
def split_data(df, test_size=0.3, seed=RANDOM_SEED):
    train_df, test_df = train_test_split(
        df,
        test_size=test_size,
        stratify=df['category'],
        random_state=seed
    )
    print(f"Train : {len(train_df):,} samples")
    print(f"Test  : {len(test_df):,} samples")
    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)


In [74]:
def build_few_shot_examples(train_df, n_per_class=1, seed=RANDOM_SEED):
    """ดึง n ตัวอย่างต่อ class จาก train set ใส่ใน prompt"""
    return (
        train_df.groupby('category', group_keys=False)
        .apply(lambda x: x.sample(min(len(x), n_per_class), random_state=seed))
        .reset_index(drop=True)
    )

In [75]:
def classify_with_llm(headline: str, short_description: str,
                      categories: list, few_shot_df: pd.DataFrame,
                      max_retries=3) -> str:
    """Classify a single article — returns predicted category string"""
    categories_str = ", ".join(categories)

    # สร้าง few-shot block
    examples_text = ""
    for _, row in few_shot_df.iterrows():
        examples_text += (
            f"\nHeadline: {row['headline']}\n"
            f"Description: {row['short_description']}\n"
            f"Category: {row['category']}\n---"
        )

    system_prompt = (
        f"You are a news article category classifier.\n"
        f"Classify the article into exactly one of these categories:\n{categories_str}\n\n"
        f"Rules:\n"
        f"- Respond with ONLY the exact category name\n"
        f"- No explanation, no punctuation, no extra words"
    )

    user_prompt = (
        f"Examples:{examples_text}\n\n"
        f"Now classify this article:\n"
        f"Headline: {headline}\n"
        f"Description: {short_description}\n"
        f"Category:"
    )

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user",   "content": user_prompt}
                ],
                temperature=0,
                max_tokens=20
            )
            predicted = response.choices[0].message.content.strip()

            # Exact match (case-insensitive)
            for cat in categories:
                if cat.lower() == predicted.lower():
                    return cat

            # Partial / substring match
            predicted_upper = predicted.upper()
            for cat in categories:
                if cat.upper() in predicted_upper or predicted_upper in cat.upper():
                    return cat

            # ถ้าหาไม่เจอ return ดิบ (จะถูก count ว่า wrong)
            return predicted

        except Exception as e:
            print(f"  ⚠ API error (attempt {attempt+1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                time.sleep(2)

    return "UNKNOWN"

In [76]:
def evaluate(test_df: pd.DataFrame, train_df: pd.DataFrame, categories: list):
    few_shot_df = build_few_shot_examples(train_df, n_per_class=1)
    y_true, y_pred = [], []
    total = len(test_df)

    print(f"\n{'='*70}")
    print(f" Evaluating {total} test samples  |  Model: {MODEL}")
    print(f"{'='*70}")

    for i, (_, row) in enumerate(test_df.iterrows()):
        true_label = row['category']
        pred_label = classify_with_llm(
            row['headline'],
            row['short_description'],
            categories,
            few_shot_df
        )
        y_true.append(true_label)
        y_pred.append(pred_label)

        correct_so_far = sum(t == p for t, p in zip(y_true, y_pred))
        running_acc = correct_so_far / len(y_true)
        status = "✓" if pred_label == true_label else "✗"

        print(
            f"[{i+1:3d}/{total}] {status} "
            f"True: {true_label:<22} "
            f"Pred: {pred_label:<22} "
            f"Acc: {running_acc:.2%}"
        )

        time.sleep(0.3)  # เว้นระยะ ป้องกัน rate limit

    return y_true, y_pred

In [82]:
def print_results(y_true, y_pred, categories):
    accuracy = accuracy_score(y_true, y_pred)
    correct = sum(t == p for t, p in zip(y_true, y_pred))

    print(f"\n{'='*70}")
    print(" FINAL EVALUATION RESULTS")
    print(f"{'='*70}")
    print(f"  Model         : {MODEL}")
    print(f"  Test Samples  : {len(y_true)}")
    print(f"  Correct       : {correct}")
    print(f"  Wrong         : {len(y_true) - correct}")
    print(f"  Accuracy      : {accuracy:.4f}  ({accuracy*100:.2f}%)")

    print(f"\n--- Per-Class Classification Report ---")
    print(classification_report(y_true, y_pred, labels=categories, zero_division=0))

    print("--- Confusion Matrix ---")
    cm = confusion_matrix(y_true, y_pred, labels=categories)
    cm_df = pd.DataFrame(cm, index=categories, columns=categories)
    print(cm_df.to_string())
    print(f"{'='*70}")


In [78]:
def main():
    print("="*70)
    print(f"  News Category Classifier  |  {MODEL}")
    print("="*70 + "\n")

    # 1. Load & sample from real dataset
    df = load_data(DATA_PATH, n_per_class=N_SAMPLES_PER_CLASS)
    categories = sorted(df['category'].unique().tolist())

    # 2. Stratified train/test split
    train_df, test_df = split_data(df, test_size=0.3)

    print(f"\nTest set distribution:")
    print(test_df['category'].value_counts().to_string())

    # 3. Run LLM classification on test set
    y_true, y_pred = evaluate(test_df, train_df, categories)

    # 4. Print & save results
    print_results(y_true, y_pred, categories)


if __name__ == "__main__":
    main()

  News Category Classifier  |  gemini-2.5-flash-lite

Loading dataset from:
  D:\Seminar_Project\dataset\News_Category_Dataset_v3.json
Total records : 209,527
Columns       : ['link', 'headline', 'category', 'short_description', 'authors', 'date']
Categories    : 42 classes
  ['ARTS', 'ARTS & CULTURE', 'BLACK VOICES', 'BUSINESS', 'COLLEGE', 'COMEDY', 'CRIME', 'CULTURE & ARTS', 'DIVORCE', 'EDUCATION', 'ENTERTAINMENT', 'ENVIRONMENT', 'FIFTY', 'FOOD & DRINK', 'GOOD NEWS', 'GREEN', 'HEALTHY LIVING', 'HOME & LIVING', 'IMPACT', 'LATINO VOICES', 'MEDIA', 'MONEY', 'PARENTING', 'PARENTS', 'POLITICS', 'QUEER VOICES', 'RELIGION', 'SCIENCE', 'SPORTS', 'STYLE', 'STYLE & BEAUTY', 'TASTE', 'TECH', 'THE WORLDPOST', 'TRAVEL', 'U.S. NEWS', 'WEDDINGS', 'WEIRD NEWS', 'WELLNESS', 'WOMEN', 'WORLD NEWS', 'WORLDPOST']



C:\Users\NBODT\AppData\Local\Temp\ipykernel_30204\1598463605.py:18: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), n_per_class * 3), random_state=seed))
C:\Users\NBODT\AppData\Local\Temp\ipykernel_30204\2845922991.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), n_per_class), random_state=seed))


Sampled records: 882 (21 per class max)

Train : 617 samples
Test  : 265 samples

Test set distribution:
category
SPORTS            7
IMPACT            7
WELLNESS          7
TRAVEL            7
ENTERTAINMENT     7
BUSINESS          7
PARENTS           7
CULTURE & ARTS    7
MEDIA             7
WEIRD NEWS        7
WORLDPOST         7
GOOD NEWS         7
PARENTING         7
POLITICS          6
WEDDINGS          6
TECH              6
STYLE & BEAUTY    6
FOOD & DRINK      6
TASTE             6
CRIME             6
DIVORCE           6
RELIGION          6
STYLE             6
GREEN             6
WORLD NEWS        6
U.S. NEWS         6
HEALTHY LIVING    6
BLACK VOICES      6
FIFTY             6
ARTS & CULTURE    6
MONEY             6
SCIENCE           6
COMEDY            6
ARTS              6
QUEER VOICES      6
WOMEN             6
THE WORLDPOST     6
COLLEGE           6
LATINO VOICES     6
ENVIRONMENT       6
HOME & LIVING     6
EDUCATION         6

 Evaluating 265 test samples  |  Model: deeps